# MAL Model Experiments

Use this notebook to compare candidate models without putting experimental training code in the FastAPI import path. Production training and inference helpers live in `app/model.py`; this notebook imports those helpers and only keeps exploratory orchestration here.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

repo_or_mal_dir = Path.cwd()
mal_dir = repo_or_mal_dir if (repo_or_mal_dir / "app").exists() else repo_or_mal_dir / "MAL"
sys.path.insert(0, str(mal_dir))

from app.model import (  # noqa: E402
    DATASET_PATH,
    FEATURE_COLUMNS,
    MODEL_PATH,
    RANDOM_STATE,
    TARGET_COLUMN,
    evaluate_model,
    load_dataset,
    save_model,
    train_validation_test_split,
)

In [ ]:
df = load_dataset(DATASET_PATH)

display(df.head())
print(f"Rows: {len(df):,}")
print(f"Features: {FEATURE_COLUMNS}")
display(df[TARGET_COLUMN].value_counts().sort_index().rename("rows_per_rating"))

In [ ]:
x_train, x_validation, x_test, y_train, y_validation, y_test = train_validation_test_split(df)

print(f"Train rows: {len(x_train):,}")
print(f"Validation rows: {len(x_validation):,}")
print(f"Test rows: {len(x_test):,}")

In [ ]:
candidate_models = {
    "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=3),
    "random_forest": RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100),
    "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=100),
}

results = []
trained_models = {}

for name, model in candidate_models.items():
    model.fit(x_train, y_train)
    trained_models[name] = model
    validation_metrics = evaluate_model(model, x_validation, y_validation)
    test_metrics = evaluate_model(model, x_test, y_test)
    results.append(
        {
            "model": name,
            "validation_accuracy": validation_metrics["accuracy"],
            "validation_macro_f1": validation_metrics["macro_f1"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
        }
    )

metrics_df = pd.DataFrame(results).sort_values("validation_macro_f1", ascending=False)
display(metrics_df)

In [ ]:
best_model_name = metrics_df.iloc[0]["model"]
best_model = trained_models[best_model_name]

print(f"Best validation model: {best_model_name}")

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLUMNS)
    display(importances.sort_values(ascending=False).rename("importance"))

In [ ]:
# Uncomment when you deliberately want to replace the production artifact.
# save_model(best_model, MODEL_PATH)
# print(f"Saved {best_model_name} to {MODEL_PATH}")